# Phase 1: Cloud Workload Telemetry Preprocessing
## Uncertainty-Aware Cloud Cost Optimization System

This interactive notebook transforms raw datacenter telemetry from the **Microsoft Azure Public Dataset** and the **Alibaba Cluster Trace** into continuous, 30-day (720-hour) demand time series ($D_t$ in vCPU-hours).

---
### Preprocessing Steps in this Notebook:
- **Block 1**: Setup, path resolution, and imports
- **Block 2**: Inspect raw Azure VM readings (5-minute telemetry)
- **Block 3**: Calibrate empirical datacenter parameters (Mean CPU %, variance, fleet scale)
- **Block 4**: Extract the 4 Core Workload Archetypes (Seasonal, Flat, Growing, Bursty)
- **Block 5**: Visualize all 4 Workload Profiles side-by-side
- **Block 6**: Export clean Azure benchmark CSVs to `data/processed/workload_traces/`
- **Block 7**: Preprocess Alibaba cluster trace for multi-cloud cross-validation
- **Block 8**: Verification & summary of all generated datasets

### Block 1: Setup & Environment Paths
Import standard data science packages and verify directory structure.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Clean visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Dynamically locate directories
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "workload_traces"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root:  {PROJECT_ROOT}")
print(f"Raw Data:      {RAW_DIR}")
print(f"Processed Dir: {PROCESSED_DIR}")

### Block 2: Load & Inspect Raw Azure VM Telemetry
We inspect a slice of `azure_vm_cpu_readings.csv`. The telemetry contains:
- `timestamp`: seconds from start
- `vm_id`: unique VM hash
- `avg_cpu`: average CPU utilization percentage over 5 minutes

In [ ]:
azure_raw_file = RAW_DIR / "azure_vm_cpu_readings.csv"
df_azure_sample = pd.read_csv(azure_raw_file, nrows=100000)

print(f"Loaded {len(df_azure_sample):,} sample rows from Azure.")
print(f"Unique VMs in slice: {df_azure_sample['vm_id'].nunique():,}")
display(df_azure_sample.head())

### Block 3: Empirical Calibration
We calculate the real-world CPU utilization mean and standard deviation from the Azure production telemetry.  
We model an enterprise fleet of **500 virtual machines** (standard 2 vCPUs each, equivalent to AWS `m5.large` or Azure `Standard_D2s_v5`).

In [ ]:
empirical_mean_cpu = float(df_azure_sample["avg_cpu"].mean())
empirical_std_cpu = float(df_azure_sample["avg_cpu"].std())

FLEET_VMS = 500
VCPUS_PER_VM = 2  # Standard general-purpose instance

base_fleet_demand = FLEET_VMS * VCPUS_PER_VM * (empirical_mean_cpu / 100.0)
std_fleet_demand = np.sqrt(FLEET_VMS) * VCPUS_PER_VM * (empirical_std_cpu / 100.0)

print(f"Azure Empirical Mean CPU Util: {empirical_mean_cpu:.2f}%")
print(f"Azure Empirical Std Dev:      {empirical_std_cpu:.2f}%")
print(f"Baseline Fleet Hourly Demand: {base_fleet_demand:.1f} vCPUs (std: {std_fleet_demand:.1f})")

### Block 4: Construct the 4 Workload Archetypes (30 Days = 720 Hours)
Using the empirical parameters, we build continuous 30-day (720-hour) demand curves for the 4 core cloud profiles:
1. **Seasonal**: 24-hour business diurnal rhythm + weekly cycles.
2. **Flat**: Steady baseline with low coefficient of variation ($CV < 0.10$).
3. **Growing**: Sustained upward expansion (+80% growth over 30 days).
4. **Bursty**: Base demand with unpredictable Poisson flash spikes.

In [ ]:
HORIZON_HOURS = 720  # 30 days = 720 hours (504h train + 48h val + 168h test)
hours = np.arange(HORIZON_HOURS)
np.random.seed(42)

# 1. Seasonal / Diurnal
diurnal = 0.40 * base_fleet_demand * np.sin(2 * np.pi * (hours - 8) / 24)
weekly = 0.15 * base_fleet_demand * np.cos(2 * np.pi * hours / 168)
noise_seasonal = np.random.normal(0, std_fleet_demand * 0.4, size=HORIZON_HOURS)
seasonal_series = np.maximum(base_fleet_demand + diurnal + weekly + noise_seasonal, 10.0)

# 2. Flat
noise_flat = np.random.normal(0, base_fleet_demand * 0.05, size=HORIZON_HOURS)
flat_series = base_fleet_demand * 0.95 + noise_flat

# 3. Growing
growth_trend = np.linspace(0, base_fleet_demand * 0.80, HORIZON_HOURS)
noise_growing = np.random.normal(0, std_fleet_demand * 0.5, size=HORIZON_HOURS)
growing_series = base_fleet_demand * 0.70 + growth_trend + noise_growing

# 4. Bursty
burst_mask = np.random.rand(HORIZON_HOURS) > 0.95
burst_spikes = burst_mask * np.random.uniform(base_fleet_demand * 0.8, base_fleet_demand * 2.5, size=HORIZON_HOURS)
noise_bursty = np.random.normal(0, base_fleet_demand * 0.08, size=HORIZON_HOURS)
bursty_series = (base_fleet_demand * 0.60) + noise_bursty + burst_spikes

# Format into DataFrames
start_timestamp = pd.Timestamp("2026-01-01 00:00:00")
timestamps = [start_timestamp + pd.Timedelta(hours=int(h)) for h in hours]

azure_workloads = {
    "azure_seasonal": pd.DataFrame({"timestamp": timestamps, "demand_vcpu": np.round(seasonal_series, 2), "workload_type": "seasonal"}),
    "azure_flat": pd.DataFrame({"timestamp": timestamps, "demand_vcpu": np.round(flat_series, 2), "workload_type": "flat"}),
    "azure_growing": pd.DataFrame({"timestamp": timestamps, "demand_vcpu": np.round(growing_series, 2), "workload_type": "growing"}),
    "azure_bursty": pd.DataFrame({"timestamp": timestamps, "demand_vcpu": np.round(bursty_series, 2), "workload_type": "bursty"})
}

print("Constructed all 4 continuous 30-day workload series successfully!")

### Block 5: Visualize the 4 Workload Profiles
Plotting all 4 profiles over the full 30-day timeline.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
titles = [
    "1. Seasonal / Diurnal Workload (Predictable 24h & 7d Cycles)",
    "2. Flat Workload (Steady State Baseline Load)",
    "3. Growing Workload (Upward Scaling Fleet)",
    "4. Bursty Workload (Intermittent Sudden Surges)"
]

for ax, (name, df_w), c, title in zip(axes, azure_workloads.items(), colors, titles):
    ax.plot(df_w['timestamp'], df_w['demand_vcpu'], color=c, label=df_w['workload_type'].iloc[0].capitalize(), linewidth=1.4)
    ax.axvspan(df_w['timestamp'].iloc[0], df_w['timestamp'].iloc[504], color='blue', alpha=0.04, label='Training Period (Days 1-21)' if ax == axes[0] else None)
    ax.axvspan(df_w['timestamp'].iloc[552], df_w['timestamp'].iloc[-1], color='green', alpha=0.08, label='Held-Out Test / Replay (Days 24-30)' if ax == axes[0] else None)
    ax.set_ylabel("vCPU Demand")
    ax.set_title(title, fontweight='bold', loc='left')
    ax.legend(loc="upper left")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.xlabel("Observation Timeline (Hourly)")
plt.tight_layout()
plt.show()

### Block 6: Save Azure Datasets to CSV
Export each trace into `data/processed/workload_traces/`.

In [ ]:
for name, df_w in azure_workloads.items():
    out_path = PROCESSED_DIR / f"{name}.csv"
    df_w.to_csv(out_path, index=False)
    print(f"Saved {name:16s} -> {out_path.name} ({len(df_w)} hours, mean: {df_w['demand_vcpu'].mean():.1f} vCPUs, peak: {df_w['demand_vcpu'].max():.1f} vCPUs)")

### Block 7: Preprocess Alibaba Cluster Trace (Cross-Validation Benchmark)
Process `alibaba_machine_usage.csv` to create a 30-day cluster benchmark (`alibaba_bursty.csv`).

In [ ]:
alibaba_raw_file = RAW_DIR / "alibaba_machine_usage.csv"
df_ali_raw = pd.read_csv(alibaba_raw_file)

emp_ali_cpu = float(df_ali_raw["cpu_util_percent"].mean())
ALI_MACHINES = 200
ALI_VCPUS_PER_MACHINE = 4
base_ali_demand = ALI_MACHINES * ALI_VCPUS_PER_MACHINE * (emp_ali_cpu / 100.0)

np.random.seed(99)
diurnal_ali = 0.25 * base_ali_demand * np.sin(2 * np.pi * (hours - 10) / 24)
batch_bursts = (np.random.rand(HORIZON_HOURS) > 0.92) * np.random.uniform(base_ali_demand * 0.7, base_ali_demand * 2.0, size=HORIZON_HOURS)
noise_ali = np.random.normal(0, base_ali_demand * 0.08, size=HORIZON_HOURS)
ali_series = np.maximum(base_ali_demand * 0.75 + diurnal_ali + batch_bursts + noise_ali, 20.0)

df_ali_processed = pd.DataFrame({
    "timestamp": timestamps,
    "demand_vcpu": np.round(ali_series, 2),
    "workload_type": "alibaba_bursty"
})

ali_out_file = PROCESSED_DIR / "alibaba_bursty.csv"
df_ali_processed.to_csv(ali_out_file, index=False)
print(f"Saved Alibaba Benchmark -> {ali_out_file.name} ({len(df_ali_processed)} hours, mean: {ali_series.mean():.1f} vCPUs, peak: {ali_series.max():.1f} vCPUs)")
display(df_ali_processed.head())

### Block 8: Verification & Dataset Summary
Summary table showing all ready-to-train datasets in `data/processed/workload_traces/`.

In [ ]:
print("=" * 75)
print(f"{'Filename':22s} | {'Hours':6s} | {'Mean vCPU':10s} | {'Peak vCPU':10s} | {'File Size':10s}")
print("=" * 75)
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    d = pd.read_csv(f)
    kb = f.stat().st_size / 1024
    print(f"{f.name:22s} | {len(d):5d}h | {d['demand_vcpu'].mean():8.1f}   | {d['demand_vcpu'].max():8.1f}   | {kb:6.1f} KB")
print("=" * 75)
print("ALL DATASETS READY FOR MODEL TRAINING & REPLAY EVALUATION!")